In [1]:
import pandas as pd
import json

In [2]:
with open("./raw/mongo_orders.json", "r") as f:
    data = json.load(f)

items = pd.json_normalize(
    data,
    record_path="items",
    meta=[
        '_id',
        'order_id',
        'order_date',
        'status',
        'amount',
        'items',
        'payment',
        'updated_at',
        'customer',
        'customer_id',
        'coupon',
        'delivery_notes'
    ]
)

app = pd.DataFrame(items)

app.head()

,category,quantity,_id,order_id,order_date,status,amount,items,payment,updated_at,customer,customer_id,coupon,delivery_notes
0,Electronics,1,6ae8c4a71dc54dd2c64a613b,287309,2025-01-02T04:28:40,cancelled,13034.4,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'card'},1735886980000,None,1553.0,ELEVEN11,NaN
1,Electronics,1,6ae8c4a71dc54dd2c64a6151,133843,2025-01-04T01:23:47,pending,10997.53,"[{'category': 'Electronics', 'quantity': 1}]",{'method': 'cash_on_delivery'},1735985387000,None,496.0,NaN,NaN
2,Electronics,3,6ae8c4a71dc54dd2c64a615d,126946,2025-01-05T18:26:56,delivered,42176.02,"[{'category': 'Electronics', 'quantity': 3}]",{'method': 'bank_transfer'},1736383256000,"{'id': 1920, 'name': 'Hassan Mirza', 'city': '...",None,NaN,NaN
3,Books,1,6ae8c4a71dc54dd2c64a615e,162981,2025-01-05T22:55:02,delivered,446.09,"[{'category': 'Books', 'quantity': 1}]",{'method': 'card'},1736569322000,"{'id': 43, 'name': 'Saad Siddiqui', 'city': 'L...",None,NaN,NaN
4,Grocery,1,6ae8c4a71dc54dd2c64a6163,294333,2025-01-06T06:28:47,delivered,5856.5,"[{'category': 'Grocery', 'quantity': 1}]",{'method': 'cash_on_delivery'},1736422127000,None,1176.0,NaN,NaN


In [3]:
web = pd.read_csv("./raw/orders.csv")
web.columns

Index(['order_id', 'customer_id', 'order_date', 'category', 'quantity',
       'amount', 'status', 'payment_method', 'updated_at'],
      dtype='str')

In [4]:
def setCustomerId(data):
    # print(data['customer'])
    if data['customer_id'] and int(data['customer_id']) >= 0:
        return int(data['customer_id'])
    else:
        return data["customer"]["id"]

In [5]:
app["customer_id"] = app[["customer_id", "customer"]].apply(lambda x: setCustomerId(x), axis=1)

In [6]:
app.drop(columns=["customer"], inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_58922/2104764804.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  app.drop(columns=["customer"], inplace=True)


In [7]:
def setItemsData(data, name):
    return data[0][name]

In [8]:
app["category"] = app["items"].apply(lambda x: setItemsData(x, "category"))
app["quantity"] = app["items"].apply(lambda x: setItemsData(x, "quantity"))

In [9]:
def setPaymentMethod(data):
    return data['method']

In [10]:
app["payment_method"] = app["payment"].apply(lambda x: setPaymentMethod(x))

In [11]:
app.drop(columns=["items", "payment", "_id", "coupon", "delivery_notes"], inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_58922/4175591087.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  app.drop(columns=["items", "payment", "_id", "coupon", "delivery_notes"], inplace=True)


In [12]:
app.head()
# ['order_id', 'customer_id', 'order_date', 'category', 'quantity', 'amount', 'status', 'payment_method', 'updated_at']

,category,quantity,order_id,order_date,status,amount,updated_at,customer_id,payment_method
0,Electronics,1,287309,2025-01-02T04:28:40,cancelled,13034.4,1735886980000,1553,card
1,Electronics,1,133843,2025-01-04T01:23:47,pending,10997.53,1735985387000,496,cash_on_delivery
2,Electronics,3,126946,2025-01-05T18:26:56,delivered,42176.02,1736383256000,1920,bank_transfer
3,Books,1,162981,2025-01-05T22:55:02,delivered,446.09,1736569322000,43,card
4,Grocery,1,294333,2025-01-06T06:28:47,delivered,5856.5,1736422127000,1176,cash_on_delivery


In [13]:
orders = pd.concat([app, web], ignore_index=True)

In [14]:
len(orders)

205000

In [15]:
orders.duplicated().sum()

np.int64(0)

In [16]:
orders.duplicated(subset=["order_id", "customer_id", "category", "quantity", "amount", 'status', "payment_method"]).sum()

np.int64(5000)

In [17]:
orders.loc[orders.duplicated(subset=["order_id", "customer_id", "category", "quantity", "amount", "status", "payment_method"], keep=False)]

,category,quantity,order_id,order_date,status,amount,updated_at,customer_id,payment_method
0,Electronics,1,287309,2025-01-02T04:28:40,cancelled,13034.4,1735886980000,1553,card
1,Electronics,1,133843,2025-01-04T01:23:47,pending,10997.53,1735985387000,496,cash_on_delivery
2,Electronics,3,126946,2025-01-05T18:26:56,delivered,42176.02,1736383256000,1920,bank_transfer
3,Books,1,162981,2025-01-05T22:55:02,delivered,446.09,1736569322000,43,card
4,Grocery,1,294333,2025-01-06T06:28:47,delivered,5856.5,1736422127000,1176,cash_on_delivery
...,...,...,...,...,...,...,...,...,...
204510,Home,1,186809,2026-10-23,shipped,18735.0,2026-10-24 23:59:00,1610,cash_on_delivery
204543,Electronics,1,116292,2026-10-23,shipped,87191.14,2026-10-24 23:59:00,520,bank_transfer
204547,Fashion,1,173292,2026-10-23,delivered,2487.6,2026-10-24 23:59:00,696,card
204570,Electronics,1,299374,2026-10-23,cancelled,41375.88,2026-10-24 23:59:00,1146,wallet


In [18]:
orders.loc[orders["order_id"] == 287309]

,category,quantity,order_id,order_date,status,amount,updated_at,customer_id,payment_method
0,Electronics,1,287309,2025-01-02T04:28:40,cancelled,13034.4,1735886980000,1553,card
5325,Electronics,1,287309,2025-01-02,cancelled,13034.4,2025-01-03 06:49:40,1553,card


# Handling Duplicates
I will remove them all because when i check for duplicates using order id, customer id, category, quantity, amount, payment method and status thats mean the orders are duplicated

In [19]:
orders.drop_duplicates(subset=["order_id", "customer_id", "category", "quantity", "amount", "status", "payment_method"], keep="first", inplace=True)

/var/folders/41/49dnj5213l1f6btjl3b5rdlh0000gn/T/ipykernel_58922/1311637366.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  orders.drop_duplicates(subset=["order_id", "customer_id", "category", "quantity", "amount", "status", "payment_method"], keep="first", inplace=True)


In [20]:
len(orders)

200000

In [21]:
orders.info()

<class 'pandas.DataFrame'>
Index: 200000 entries, 0 to 204999
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   category        200000 non-null  str   
 1   quantity        200000 non-null  int64 
 2   order_id        200000 non-null  object
 3   order_date      200000 non-null  str   
 4   status          200000 non-null  str   
 5   amount          200000 non-null  object
 6   updated_at      200000 non-null  object
 7   customer_id     200000 non-null  int64 
 8   payment_method  200000 non-null  str   
dtypes: int64(2), object(3), str(4)
memory usage: 22.0+ MB


In [22]:
orders["order_date"] = pd.to_datetime(orders["order_date"], format="mixed")
orders["updated_at"] = pd.to_datetime(orders["updated_at"], format="mixed")

In [23]:
orders.loc[orders.amount <= 0]

,category,quantity,order_id,order_date,status,amount,updated_at,customer_id,payment_method


In [24]:
orders.dtypes

category                     str
quantity                   int64
order_id                  object
order_date        datetime64[us]
status                       str
amount                    object
updated_at        datetime64[ns]
customer_id                int64
payment_method               str
dtype: object

In [25]:
orders.isnull().sum()

category          0
quantity          0
order_id          0
order_date        0
status            0
amount            0
updated_at        0
customer_id       0
payment_method    0
dtype: int64

In [26]:
orders.duplicated().sum()

np.int64(0)

In [27]:
len(orders)

200000

Order Columns donot have any null, all Data types are aligned an Duplicates are remove after Verification

In [29]:
import os

In [30]:
os.makedirs("./cleaned", exist_ok=True)
orders.to_csv("./cleaned/cleaned_orders.csv", index=False)